# Hope: GatedTCN Model V4 - Advanced Training

> **Instructions:** Run all cells sequentially from top to bottom in a fresh runtime. Do not skip cells or re-run individual cells out of order.


In [ ]:
# Environment Detection and Path Setup
import os
import sys
import requests

def get_env():
    # Colab check
    try:
        resp = requests.get('http://metadata.google.internal/computeMetadata/v1/instance/', 
                            headers={'Metadata-Flavor': 'Google'}, timeout=2)
        if resp.status_code == 200: return 'colab'
    except: pass
    
    # Kaggle check
    if os.path.exists('/kaggle/input') and os.path.ismount('/kaggle/input'): return 'kaggle'
    
    return None

env = get_env()
if env == 'colab':
    data_path = '/content/ticks.csv'
    if '/content/scripts' not in sys.path: sys.path.append('/content/scripts')
    from google.colab import drive
    try: drive.mount('/content/drive', force_remount=False)
    except Exception as _e: print(f'WARNING: Google Drive not mounted ({_e})')
elif env == 'kaggle':
    data_path = '/kaggle/input/ticks-csv/ticks.csv'
    if '/kaggle/working/scripts' not in sys.path: sys.path.append('/kaggle/working/scripts')
else:
    if os.environ.get('FORCE_CLOUD_ENV') == '1':
        env = 'manual'
        data_path = 'data/ticks.csv'
    else:
        raise RuntimeError('Local training execution is prohibited. Run on Colab/Kaggle.')

print(f'Running on: {env}')
print(f'Data path: {data_path}')

# Auto-clone logic for cross-platform compatibility
if not os.path.exists('scripts/hope_ml'):
    print('Fetching repository scripts...')
    !git clone https://github.com/planetazul3/hope.git hope_tmp
    !cp -r hope_tmp/scripts ./
    !rm -rf hope_tmp
    print('Scripts synchronized.')


In [ ]:
!pip install torch==2.1.0 onnx==1.15.0 onnxruntime==1.16.3 numpy==1.24.3 pandas==2.1.1 scikit-learn==1.3.1 tqdm matplotlib seaborn


In [ ]:
import torch
import torch.nn as nn
import torch.onnx
import numpy as np
import pandas as pd
import math
import os
import copy
import shutil
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, precision_recall_curve, auc as pr_auc
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import LambdaLR

try:
    from hope_ml.common import GatedTCNV4, prepare_features, contrastive_loss, focal_loss, block_mask
    print("Successfully imported hope_ml.common")
except ImportError as e:
    print(f"Import failed: {e}. Ensure scripts directory is in path.")


In [ ]:
import random
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print(f"Global random seed set to {SEED}")


In [ ]:
def load_data_from_csv(csv_path, limit=None):
    search_paths = [csv_path, "data/ticks.csv", "/content/ticks.csv", "/kaggle/input/ticks-csv/ticks.csv"]
    actual_path = None
    for p in search_paths:
        if os.path.exists(p):
            actual_path = p
            break
    
    if actual_path is None:
        print("CSV not found in common paths.")
        return None
    
    print(f"Loading data from: {actual_path}")
    df = pd.read_csv(actual_path, header=None, names=['epoch', 'quote'], nrows=limit if limit is not None else None)
    return df['quote'].values.astype(np.float32)


In [ ]:
csv_path = data_path
seq_len = 32
input_dim = 8

prices = load_data_from_csv(csv_path)
if prices is not None:
    x_all, y_dir_all, y_vol_all = prepare_features(prices, seq_len=seq_len)
else:
    print("Using synthetic data for dry run.")
    x_all = torch.randn(2000, seq_len, input_dim)
    y_dir_all = torch.randint(0, 2, (2000, 1)).float()
    y_vol_all = torch.rand(2000, 1)

# Temporal Split with seq_len gap to prevent leakage
split = int(len(x_all) * 0.8)
train_ds = TensorDataset(x_all[:split], y_dir_all[:split], y_vol_all[:split])
val_ds = TensorDataset(x_all[split + seq_len:], y_dir_all[split + seq_len:], y_vol_all[split + seq_len:])
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GatedTCNV4(input_dim=input_dim).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)

# Phase 1: Contrastive Pre-training
print(f"Starting Phase 1: Contrastive Pre-training... (Initial LR: {optimizer.param_groups[0]['lr']:.6f})")
contrastive_epochs = 5
for epoch in range(contrastive_epochs):
    model.train()
    total_cl_loss = 0
    for bx, _, _ in train_loader:
        bx = bx.to(device)
        bx_aug1, bx_aug2 = block_mask(bx), block_mask(bx)
        optimizer.zero_grad()
        f1, f2 = model(bx_aug1, return_feat=True), model(bx_aug2, return_feat=True)
        loss = contrastive_loss(f1, f2, temperature=0.1)
        loss.backward()
        optimizer.step()
        total_cl_loss += loss.item()
    print(f"Pre-train Epoch {epoch}, CL Loss: {total_cl_loss/len(train_loader):.4f}")

# Phase 2: Supervised Fine-tuning with AMP
print("Starting Phase 2: Supervised Fine-tuning with AMP...")
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
warmup_epochs = 5
def lr_lambda(epoch): return (epoch + 1) / warmup_epochs if epoch < warmup_epochs else 1.0
warmup_scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda)
plateau_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

num_pos = torch.sum(y_dir_all[:split]).item()
pos_weight = (split - num_pos) / num_pos if num_pos > 0 else 1.0
pos_weight_t = torch.tensor([pos_weight], dtype=torch.float32, device=device)

scaler = torch.cuda.amp.GradScaler()
best_auc = 0.0
history = {'train_loss': [], 'val_auc': [], 'val_prauc': []}

for epoch in range(20):
    model.train()
    total_loss = 0
    for bx, by_dir, by_vol in train_loader:
        bx, by_dir, by_vol = bx.to(device), by_dir.to(device), by_vol.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            out_dir, out_vol = model(bx)
            l_dir = focal_loss(out_dir, by_dir, pos_weight_t)
            l_vol = nn.functional.mse_loss(out_vol, by_vol)
            loss = l_dir + 0.2 * l_vol
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()

    model.eval()
    v_probs, v_targets = [], []
    with torch.no_grad():
        for bx, by_dir, _ in val_loader:
            out_dir, _ = model(bx.to(device))
            v_probs.append(out_dir)
            v_targets.append(by_dir)
    
    vp = torch.cat(v_probs).cpu().numpy().flatten()
    vt = torch.cat(v_targets).cpu().numpy().flatten()
    auc_val = roc_auc_score(vt, vp)
    p_curve, r_curve, _ = precision_recall_curve(vt, vp)
    prauc_val = pr_auc(r_curve, p_curve)
    
    if epoch < warmup_epochs: warmup_scheduler.step()
    else: plateau_scheduler.step(auc_val)
    
    history['train_loss'].append(total_loss/len(train_loader))
    history['val_auc'].append(auc_val)
    history['val_prauc'].append(prauc_val)
    
    print(f"Epoch {epoch}, AUC: {auc_val:.4f}, PR-AUC: {prauc_val:.4f}, Loss: {history['train_loss'][-1]:.4f}")
    
    if auc_val > best_auc:
        best_auc = auc_val
        torch.save(model.state_dict(), "best_model.pth")
        patience_counter = 0
    elif (patience_counter := patience_counter + 1) >= 5: break


In [ ]:
model.load_state_dict(torch.load('best_model.pth'))
model.eval()
class InferenceModel(nn.Module):
    def __init__(self, trained_model): super().__init__(); self.m = trained_model
    def forward(self, x): direction, _ = self.m(x); return direction

dummy = torch.randn(1, 32, 8).to(device)
model_path = 'model.onnx'
torch.onnx.export(InferenceModel(model), dummy, model_path, export_params=True, opset_version=15, 
                  do_constant_folding=True, input_names=['input'], output_names=['output'])

from onnxruntime.quantization import quantize_dynamic, QuantType
quantize_dynamic(model_path, 'model_quantized.onnx', weight_type=QuantType.QInt8)
print('Deployment models exported.')

# --- Cryptographic Signing Pipeline ---
try:
    from cryptography.hazmat.primitives import serialization
    from cryptography.hazmat.primitives.asymmetric import ed25519
    
    # In production, the private key should be securely injected via Colab Secrets or Kaggle User Secrets
    # For this demonstration, we'll look for an environment variable
    key_hex = os.environ.get('MODEL_SIGNING_KEY')
    if key_hex:
        private_key = ed25519.Ed25519PrivateKey.from_private_bytes(bytes.fromhex(key_hex))
        
        for target in ['model.onnx', 'model_quantized.onnx']:
            if os.path.exists(target):
                with open(target, 'rb') as f: data = f.read()
                signature = private_key.sign(data)
                with open(target + '.sig', 'wb') as f: f.write(signature)
                print(f'Signed {target} -> {target}.sig')
    else:
        print('WARNING: MODEL_SIGNING_KEY not found. Models will not be signed.')
except ImportError:
    print('Signing skipped (cryptography library not installed)')


In [ ]:
model.load_state_dict(torch.load("best_model.pth"))
model.eval()
class InferenceModel(nn.Module):
    def __init__(self, trained_model): super().__init__(); self.m = trained_model
    def forward(self, x): direction, _ = self.m(x); return direction

dummy = torch.randn(1, 32, 8).to(device)
torch.onnx.export(InferenceModel(model), dummy, "model.onnx", export_params=True, opset_version=15, 
                  do_constant_folding=True, input_names=['input'], output_names=['output'])

from onnxruntime.quantization import quantize_dynamic, QuantType
quantize_dynamic("model.onnx", "model_quantized.onnx", weight_type=QuantType.QInt8)
print("Deployment models exported.")
